# Training and evaluating the grading model

The grading model looks at one building crop and predicts a grade from A to F. This notebook trains it and evaluates it the way all results in the study were evaluated.

**The model.**

- **Backbone:** DINOv3 ViT-S/16 [Siméoni et al. 2025], pretrained. The crop is padded to a square (letterbox, never stretched) and resized to 448 px. The feature is the CLS token concatenated with the mean of the patch tokens.
- **Fine-tuning:** the last 4 of the 12 transformer blocks are trained, the rest stay frozen. In the "frozen" variants only the head is trained, on cached features.
- **Ordinal head (CORN)** [Shi et al. 2023]: five binary tasks, "grade > A", "grade > B" … "grade > E". The predicted grade is the number of tasks with P(grade > t) > 0.5 on the chained probabilities, which keeps predictions rank-consistent. The model knows that B is closer to C than to F.
- **Class weights:** each crop's loss is weighted by N / (6 · n_grade), so rare grades count as much as common ones.
- **Training:** AdamW, two warm-up epochs and a cosine schedule, at most 40 epochs. The epoch with the best validation balanced accuracy is kept, with early stopping after 8 epochs without improvement. Three seeds are trained per setting.

**The metrics.**

- **Balanced accuracy:** the mean of the six per-grade accuracies, i.e. the sum of the row-normalised confusion-matrix diagonal / 6, so rare grades count fully.
- **Per-grade accuracy and MAE:** MAE is the mean distance in grades (C predicted as B = 1).
- **Within one grade.**
- **95% intervals:** a bootstrap over photos (2,000 resamples), keeping all crops of a photo together.

In [ ]:
# On Colab: install the package and get the configuration files.
!pip -q install "insarag-triage[roboflow] @ git+https://github.com/leeelyehezkel/insarag-damage-triage"
!git clone -q https://github.com/leeelyehezkel/insarag-damage-triage repo 2>/dev/null || true

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from insarag_triage import core as L, analysis as A, active_learning as AL, data as DATA

REPO = Path('repo') if Path('repo').exists() else Path('.')
WORK = Path('work')
dcfg = DATA.load_config(REPO / 'configs' / 'data.yaml')
folders = {s: DATA.download(dcfg, s, WORK / 'roboflow') for s in ['original', 'al_labels', 'test12', 'test3']}
m = DATA.assemble(REPO / 'data' / 'manifests' / 'labels.csv', folders)
pd.crosstab(m.split, m.label, margins=True)

## Train one model

`TEST_SET` chooses which test set the training run reports: `'test2'` (random, blind labels) or `'test1'` (the original test fold). Test 3 is kept for notebook 05. Finished seeds are skipped, so the cell can be run again after a disconnect. On a Colab T4 a fine-tuned run takes about 20–30 minutes per seed, a frozen one a few minutes.

In [ ]:
TEST_SET = 'test2'
SEEDS = [0, 1, 2]
cfg = L.Config('v3_s_partial_448', backbone='dinov3_vits16', mode='partial', img_size=448, lr_backbone=2e-5)

def frame(test_set):
    # train and val as in the final models; the chosen test set becomes 'test'
    return pd.concat([m[m.split.isin(['train', 'val'])], m[m.split == test_set].assign(split='test')], ignore_index=True)

df = frame(TEST_SET)
run_dir = WORK / 'runs' / f'{cfg.name}_{TEST_SET}'
results = []
for s in SEEDS:
    out = run_dir / f'seed{s}'
    results.append(json.loads((out / 'result.json').read_text()) if (out / 'result.json').exists() else L.run(cfg, df, s, out))
print('balanced accuracy per seed:', [round(r['test']['overall_accuracy_diag_over_6'], 3) for r in results])

## Evaluate

The three seeds are combined by averaging their probabilities P(grade > t), as in the released model, so each test crop is counted once.

In [ ]:
test = df[df.split == 'test'].reset_index(drop=True)
cum = np.mean([AL.member_cum_probs(cfg, run_dir, s, test.path) for s in SEEDS], axis=0)
pred = pd.DataFrame({'group': test.photo, 'y_true': test.y, 'y_pred': (cum > 0.5).sum(1)})
e = L.evaluate(pred.y_true, pred.y_pred)
L.plot_confusion(e, f'{cfg.name} · {TEST_SET} · {len(SEEDS)} seeds averaged'); plt.show()
A.bootstrap_ci(pred).loc[['overall_accuracy_diag_over_6', 'mae_overall_micro', 'within1_balanced']
                          + [f'per_class_accuracy:{c}' for c in L.CLASSES] + [f'per_class_mae:{c}' for c in L.CLASSES]]

## Comparing model settings

In the study, about 15 settings were compared on the original crops:

- **What was varied:** DINOv2 vs DINOv3, ViT vs ConvNeXt, small to base models, 224 to 518 px, frozen vs partly fine-tuned, linear vs small MLP head.
- **Result:** test-1 balanced accuracy stayed between 0.56 and 0.65 whatever was changed, and the best four were within 0.015 of each other on validation. Larger models did not help.
- **Conclusion:** the model was not what limited accuracy. Two other things did: the data (later found to be stretched), and how consistent the labels are (notebook 02).

The cell below runs any list of settings on the current data and puts them side by side. Frozen settings are fast because their features are cached.

In [ ]:
CONFIGS = [
    L.Config('v3_s_frozen_224', backbone='dinov3_vits16', mode='frozen', img_size=224),
    L.Config('siglip2_frozen_224', backbone='siglip2_b16', mode='frozen', img_size=224),
    L.Config('cnx_in22k_frozen_224', backbone='convnext_tiny_in22k', mode='frozen', img_size=224),
    # L.Config('v3_s_partial_448', backbone='dinov3_vits16', mode='partial', img_size=448, lr_backbone=2e-5),
    # L.Config('v3_cnxt_partial_448', backbone='dinov3_convnext_tiny', mode='partial', img_size=448, lr_backbone=5e-5),
]
rows = []
for c in CONFIGS:
    for s in SEEDS:
        out = WORK / 'runs' / f'{c.name}_{TEST_SET}' / f'seed{s}'
        r = json.loads((out / 'result.json').read_text()) if (out / 'result.json').exists() else L.run(c, df, s, out)
        rows.append(dict(model=c.name, seed=s, val_acc=r['val']['overall_accuracy_diag_over_6'],
                         test_acc=r['test']['overall_accuracy_diag_over_6'], test_mae=r['test']['mae_overall_micro'],
                         **{f'test_{k}': v for k, v in r['test']['per_class_accuracy'].items()}))
cmp = pd.DataFrame(rows).groupby('model').mean(numeric_only=True).drop(columns='seed').sort_values('val_acc', ascending=False)
cmp.round(3)

## Standing or collapsed?

Six-grade accuracy hides which mistakes are made. For rescue work, the most important question is whether a collapsed building (D–F) is ever called standing (A–C).

In [ ]:
st = pd.crosstab(np.where(pred.y_true <= 2, 'standing (A–C)', 'collapsed (D–F)'),
                 np.where(pred.y_pred <= 2, 'standing (A–C)', 'collapsed (D–F)'), rownames=['true'], colnames=['predicted'])
st

## References

Shi, X., Cao, W., & Raschka, S. (2023). Deep neural networks for rank-consistent ordinal regression based on conditional probabilities. Pattern Analysis and Applications 26, 941–955. doi:10.1007/s10044-023-01181-9 ·
Siméoni, O., et al. (2025). DINOv3. arXiv:2508.10104